# Practice Skeleton: Linear Regression with Scikit-Learn SGDRegressor

**Course context:** Optional Lab C1_W2_Lab05 (Machine Learning Specialization) – extended for practice, alternates, simulation, and responsible use.

## Goals
- Load multi-feature housing data (size, bedrooms, floors, age → price in $1000s).
- Apply z-score normalization with `StandardScaler`.
- Fit `sklearn.linear_model.SGDRegressor` (stochastic gradient descent).
- Inspect learned parameters, make predictions, evaluate, and visualize.
- Explore **alternate implementations**, extra practice drills, and a **Monte-Carlo simulation** of sensitivity to learning rate, sample size, and noise.
- Produce audience-adapted insights and understand model limitations.

## Cheat Sheet (keep this panel open while working)

| Task | Code / Concept |
|------|----------------|
| Load data | `np.loadtxt('data/houses.txt', delimiter=',')` → X = data[:, :4], y = data[:, 4] |
| Feature names | `['size(sqft)', 'bedrooms', 'floors', 'age']` |
| Normalize | `scaler = StandardScaler(); X_norm = scaler.fit_transform(X)` |
| Peak-to-peak | `np.ptp(X, axis=0)` – should shrink dramatically after scaling |
| Fit SGD | `sgdr = SGDRegressor(max_iter=1000, tol=1e-3); sgdr.fit(X_norm, y)` |
| Parameters | `w = sgdr.coef_`, `b = sgdr.intercept_` (on **normalized** scale) |
| Predict | `y_pred = sgdr.predict(X_norm)` or `X_norm @ w + b` |
| Metrics | `from sklearn.metrics import r2_score, mean_squared_error` |
| Inverse scale note | Coefficients are for z-scored features; interpret relative importance carefully |
| Common pitfall | Forgetting to scale → slow / failed convergence; scaling on full data before split (leakage) |
| Alternate OLS | `from sklearn.linear_model import LinearRegression` (closed-form, no iteration) |
| Pipeline | `make_pipeline(StandardScaler(), SGDRegressor(...))` |

---
## Flowchart of Desired Outcome

![Sklearn GD Flowchart](sklearn_gd_flowchart.png)

---
**Instructions:** Fill in every `# TODO` cell. Run the solution notebook only after you have attempted the exercises.

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline

np.set_printoptions(precision=2, suppress=True)
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries ready.")

## 1. Load the Dataset

Load `data/houses.txt` (comma-separated, no header).

- Columns 0-3 → features: size(sqft), bedrooms, floors, age
- Column 4 → target: price in $1000s

Store feature names in a list for later plotting.

In [ ]:
# TODO: load the data and split into X_train (n,4) and y_train (n,)
data = None  # np.loadtxt(...)
X_train = None
y_train = None
X_features = ['size(sqft)', 'bedrooms', 'floors', 'age']

print(f"X shape: {X_train.shape if X_train is not None else 'TODO'}, y shape: {y_train.shape if y_train is not None else 'TODO'}")

### Quick EDA
Create a DataFrame, show `.describe()`, and print correlation of each feature with price.

In [ ]:
# TODO: build df, print describe() and corr with price
df = None
print("TODO: descriptive statistics and correlations")

## 2. Scale / Normalize the Training Data

Use `StandardScaler().fit_transform`.

Print peak-to-peak range before and after. Why does SGD need this?

In [ ]:
# TODO: create scaler, fit_transform X_train → X_norm
scaler = None
X_norm = None

print("Peak-to-peak raw vs normalized – fill in after coding")

## 3. Create and Fit the SGD Regression Model

Instantiate `SGDRegressor(max_iter=1000)` (add `random_state=42` for reproducibility) and call `.fit`.

In [ ]:
# TODO: create and fit sgdr
sgdr = None

print("TODO: print model and n_iter_ / t_")

## 4. View Parameters

Extract `coef_` and `intercept_`. Compare to the reference values from the pure-Python lab:
`w ≈ [110.56, -21.27, -32.71, -37.97]`, `b ≈ 363.16`

In [ ]:
# TODO: extract w_norm, b_norm and print them next to the reference
w_norm = None
b_norm = None
print("TODO")

## 5. Make Predictions & Evaluate

- Use `sgdr.predict(X_norm)`
- Also compute `X_norm @ w_norm + b_norm` and verify they match
- Compute R², RMSE, MAE

In [ ]:
# TODO: predictions + metrics
y_pred_sgd = None
y_pred = None
print("TODO: match check and metrics")

## 6. Plot Results – Target vs Prediction by Feature

Create a 1×4 subplot. For each feature scatter the original X values against both target and prediction (different colors). Share the y-axis.

In [ ]:
# TODO: 1x4 scatter plots of target vs predict for each feature
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
# fill the loop
plt.tight_layout()
plt.show()

---
## 7. Alternate Implementations

### 7.1 Closed-form OLS with `LinearRegression`
Fit on the same `X_norm`. Compare coefficients and R² to SGD.

In [ ]:
# TODO: LinearRegression fit + compare
ols = None

### 7.2 Manual z-score + pure NumPy batch GD (optional challenge)
Implement a simple batch gradient-descent loop. Start from zero weights, use α = 0.1, 1000 iterations.

In [ ]:
# TODO (challenge): manual_zscore + gradient_descent functions
print("Optional – implement if you want deeper understanding of the GD update")

### 7.3 Pipeline
Build `make_pipeline(StandardScaler(), SGDRegressor(...))` and fit on **raw** `X_train`. This is the production-safe pattern.

In [ ]:
# TODO: pipeline
pipe = None

---
## 8. More Practice

### Practice A – New house prediction
Predict the price of a house: 1600 sqft, 3 bedrooms, 1 floor, 40 years old.
Remember to transform with the *already fitted* scaler.

In [ ]:
# TODO: new_house → transform → predict
new_house = np.array([[1600, 3, 1, 40]])
print("Predicted price ($): TODO")

### Practice B – Feature ablation
Identify the feature with the smallest absolute normalized coefficient. Drop it, re-scale, re-fit, and compare R².

In [ ]:
# TODO: ablation experiment
print("TODO")

### Practice C – Hyper-parameter grid (small)
Try a few combinations of `eta0` and `max_iter` with `learning_rate='constant'`. Record R².

In [ ]:
# TODO: small nested loop over eta0 and max_iter
print("TODO")

---
## 9. Simulation Section – Monte-Carlo Sensitivity

Write a function `simulate_once(n, sigma, eta0, max_iter)` that:
1. Generates synthetic features roughly in the house ranges,
2. Creates a linear target + Gaussian noise,
3. Scales, fits SGDRegressor,
4. Returns a dict with R², n_iter, recovered coefficients.

Then run it many times while varying noise level and sample size; plot boxplots of R².

In [ ]:
def simulate_once(n=100, sigma=20.0, eta0=0.01, max_iter=1000):
    # TODO: implement generation + fit + return dict
    return {'n': n, 'sigma': sigma, 'R2': 0.0, 'n_iter': 0}

# TODO: Monte-Carlo loops + boxplots
print("Implement simulation and visualize")

### Interactive-style parameter playground
Change the four constants below and re-run the cell to see immediate effect on R² and iterations.

In [ ]:
# === MODIFY THESE ===
SIM_N = 80
SIM_SIGMA = 25.0
SIM_ETA = 0.05
SIM_MAX_ITER = 800
# ====================

# TODO: call simulate_once with the values above and print results
print("TODO")

---
## 10. Reflection – What the Model Can / Cannot Predict

Write 4–6 bullet points (in a markdown cell or comments) summarizing:
- What linear SGD regression *can* do well on this housing task,
- What it *cannot* do (non-linearity, causality, extrapolation, uncertainty),
- At least two responsible-use practices you would follow if presenting results to a non-technical stakeholder or to a credit-risk committee.

(Consult the companion 1-page report and the Responsible-Use document for ideas.)

In [ ]:
# Your reflection notes here (or use a markdown cell)

---
## End of Practice Skeleton
Once finished, compare with the Solution notebook. Re-run the simulation with different seeds and discuss stability with a peer or mentor.